# Git + GitHub Tutorial for `palfitology`

This notebook is a step-by-step walkthrough of the full Git workflow for the `palfitology` project, from checking the repository state all the way to pushing changes up to GitHub.

**Repository:** [`nisach02/palfitology`](https://github.com/nisach02/palfitology)  
**Local path:** `~/Desktop/AntigravityProjects/palfitology`  
**Default branch:** `main`

Each code cell uses Jupyter's `!` syntax to run shell commands. You can run the cells one by one (`Shift+Enter`) and watch the output. If you'd rather run the commands in a terminal, just drop the leading `!` and run them from inside the `palfitology` directory.

---

## Workflow at a glance

1. Move into the repository.
2. Check the current state (`git status`).
3. See what changed (`git diff`).
4. Stage the files you want to commit (`git add`).
5. Commit them with a message (`git commit -m "..."`).
6. Pull anything new from GitHub (`git pull --rebase`).
7. Push your commit to GitHub (`git push`).
8. Confirm everything landed (`git log` / GitHub web UI).

We'll go through these in order.

## Step 0 — Move into the repository

Every git command operates on the repository in the current working directory, so the first thing we do is `cd` into it. In Jupyter, `%cd` (a magic) actually changes the notebook's working directory; `!cd ...` would only affect the subshell and reset immediately.

In [ ]:
%cd ~/Desktop/AntigravityProjects/palfitology

In [ ]:
# Sanity check: confirm we're in a git repo and on the right branch.
!pwd
!git rev-parse --is-inside-work-tree
!git branch --show-current

Expected output:
- `pwd` ends with `.../palfitology`
- `git rev-parse --is-inside-work-tree` prints `true`
- `git branch --show-current` prints `main` (or whatever feature branch you happen to be on)

## Step 1 — Check what changed (`git status`)

`git status` is the most-used git command. It tells you:
- Which branch you're on,
- Which files have been **modified** but not yet staged,
- Which files are **staged** (ready to be committed),
- Which files are **untracked** (new files git doesn't know about yet).

In [ ]:
!git status

The short form is handy when there are a lot of files:

In [ ]:
!git status -s

Legend for `git status -s`:

- ` M file.py` — modified, **not** staged
- `M  file.py` — modified **and** staged
- `A  file.py` — newly added, staged
- `??  file.py` — untracked (git has never seen this file before)
- `D  file.py` — deleted, staged

## Step 2 — See exactly what changed (`git diff`)

`git diff` shows the line-by-line changes in your modified files. This is the cell you want to look at carefully **before** committing — it's your last chance to catch a stray `print()`, a debug flag, or a hardcoded path before it ends up in history.

In [ ]:
# Unstaged changes (working tree vs. last commit)
!git diff

In [ ]:
# Just a summary: which files changed and by how many lines
!git diff --stat

In [ ]:
# After you've staged things, this shows what's *about* to be committed:
!git diff --staged

## Step 3 — Pull the latest from GitHub first

Before staging anything, it's a good habit to pull from `origin/main` so you're committing on top of whatever's already on GitHub. This avoids the awkward case where you commit, try to push, and discover someone (or another machine — looking at you, CEFCA cluster) pushed in the meantime.

`--rebase` keeps history linear by replaying your local commits on top of remote changes, instead of creating a merge commit.

In [ ]:
!git fetch origin

In [ ]:
# Only run this if you have NO uncommitted changes, or you'll get a complaint.
# If you do have uncommitted changes, either commit them first (Steps 4-5) or stash them:
#   !git stash
#   !git pull --rebase origin main
#   !git stash pop
!git pull --rebase origin main

## Step 4 — Stage the files you want to commit (`git add`)

Staging tells git: "these are the changes I want to include in the next commit." You can stage all changes, a single file, or even individual chunks (hunks) inside a file.

### Option A — stage everything modified or new

Convenient, but be careful: this also picks up untracked files. Make sure your `.gitignore` is doing its job (FITS cutouts, GALFIT output blocks, log files, etc.).

In [ ]:
# !git add -A

### Option B — stage a specific file

Recommended when you've touched several things and only want to commit one of them.

In [ ]:
# !git add palfitology/cli.py

### Option C — stage interactively

`-p` ("patch") walks you through each chunk of each modified file and asks `y`/`n`. Great for separating two unrelated changes that ended up in the same file.

In [ ]:
# !git add -p   # interactive — easier to run in a terminal than a notebook

Verify what's now staged:

In [ ]:
!git status -s
!echo '---'
!git diff --staged --stat

Made a mistake? `git restore --staged <file>` un-stages a file without throwing away your edits:

In [ ]:
# !git restore --staged palfitology/cli.py

## Step 5 — Commit (`git commit`)

A commit is a permanent snapshot of whatever is staged, with a message describing why the change exists.

**Commit message guidelines** (loose, but useful):
- First line ≤ 72 characters, imperative mood: *"Add sigma-cutoff detection step"*, not *"Added"* or *"Adding"*.
- Blank line, then a longer body if you need to explain *why* (the code already shows *what*).
- Reference issues/PRs if relevant.

For this tutorial we'll use a placeholder message — replace it with something real before you actually run the cell.

In [ ]:
!git commit -m "this is where you put the commit message"

### Multi-line commit messages

If you want a proper subject + body, run `git commit` with no `-m` flag — git will open your `$EDITOR`. Or chain `-m` flags (each one becomes a paragraph):

In [ ]:
# !git commit -m "this is where you put the commit message" \
#             -m "And this is where you put a longer explanation of the *why* behind the change, references to issues, etc."

### Made a typo in the message? Amend the last commit:

In [ ]:
# !git commit --amend -m "this is where you put the (corrected) commit message"

**Warning:** only amend commits you have **not yet pushed**. Amending rewrites history; if the old commit is already on GitHub, you'll need a force-push and you should know what you're doing.

## Step 6 — Look at the new commit locally

Before pushing, confirm the commit landed the way you wanted.

In [ ]:
# Last 5 commits, one line each
!git log --oneline -n 5

In [ ]:
# Show the full diff and metadata of the most recent commit
!git show HEAD

## Step 7 — Push to GitHub (`git push`)

Now we send the commit up to GitHub. The first time you push a brand-new branch, you need `-u origin <branch>` so that future `git push` / `git pull` know which remote branch you're tracking. After that, plain `git push` is enough.

In [ ]:
# For an existing branch (like main) that already tracks origin:
!git push

In [ ]:
# For a brand-new branch the first time:
# !git push -u origin my-new-feature-branch

### If the push is rejected

You'll see something like:

```
! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'github.com:nisach02/palfitology.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally.
```

Translation: somebody (or you on another machine) pushed something since your last pull. Fix:

```bash
git pull --rebase origin main
git push
```

## Step 8 — Confirm it landed on GitHub

Two ways to check.

**From the command line:** the local `main` and `origin/main` should point to the same commit hash.

In [ ]:
!git log --oneline -n 3 main
!echo '---'
!git log --oneline -n 3 origin/main

**In a browser:** open the repository, your commit should be at the top of the history.

- All commits: <https://github.com/nisach02/palfitology/commits/main>
- Repo home: <https://github.com/nisach02/palfitology>

## Bonus — Working on a feature branch instead of `main`

For anything non-trivial (a new feature, a refactor, a risky experiment) prefer a branch over committing straight to `main`. The flow is the same; you just insert a branch-creation step at the top and open a Pull Request at the bottom.

```bash
# 1. Start from up-to-date main
git checkout main
git pull --rebase origin main

# 2. Create + switch to a new branch
git checkout -b v07-fill-nan-isophote

# 3. ...edit files, then the usual cycle:
git add -p
git commit -m "this is where you put the commit message"

# 4. Push the branch for the first time
git push -u origin v07-fill-nan-isophote

# 5. Open a Pull Request from the GitHub web UI, get a review, merge.

# 6. Clean up afterward
git checkout main
git pull --rebase origin main
git branch -d v07-fill-nan-isophote
```

## Bonus — Quick troubleshooting cheatsheet

| Situation | Command |
|---|---|
| Throw away edits to one file (revert to last commit) | `git restore <file>` |
| Un-stage a file (keep edits) | `git restore --staged <file>` |
| Forgot to add a file to the last commit (not yet pushed) | `git add <file> && git commit --amend --no-edit` |
| Need to switch branches but you have unfinished edits | `git stash` … `git stash pop` |
| Want to see who changed a line | `git blame <file>` |
| Want to find which commit introduced a string | `git log -S "text" -- <file>` |
| See remote URLs | `git remote -v` |

## End-to-end — the whole flow in one cell

Once the steps above feel routine, this is the day-to-day shape of a single commit + push, in one block:

```bash
cd ~/Desktop/AntigravityProjects/palfitology
git status
git diff
git pull --rebase origin main
git add <files-you-want>
git diff --staged
git commit -m "this is where you put the commit message"
git push
git log --oneline -n 3
```

That's it. Read state → review changes → sync → stage → review staged → commit → push → confirm.